# Constraints & Keys

## Quick notes
### What this covers
- How a table protects its own data: PRIMARY KEY, FOREIGN KEY, UNIQUE, CHECK, NOT NULL, DEFAULT.
- Extras SQLite adds: generated columns, STRICT tables, WITHOUT ROWID tables.
- What ALTER TABLE can and cannot do, and the safe "rebuild the table" recipe.
### Key points
- A constraint is a rule the database enforces on every INSERT and UPDATE, no matter which app sends it.
- `INTEGER PRIMARY KEY` is special: it is an alias for the hidden `rowid`.
- Foreign keys are NOT enforced until you run `PRAGMA foreign_keys = ON` (per connection).
- `ON DELETE` / `ON UPDATE` actions decide what happens to child rows: CASCADE, SET NULL, RESTRICT, NO ACTION.
- UNIQUE allows many NULLs. CHECK passes when the result is NULL.
- `STRICT` makes SQLite enforce column types; the default is flexible typing.
### Gotchas (the part worth revising twice)
- Forgetting `PRAGMA foreign_keys = ON` means your foreign keys do nothing, silently.
- A CHECK that evaluates to NULL passes. Add `NOT NULL` if NULL must be rejected.
- `AUTOINCREMENT` is rarely needed; plain `INTEGER PRIMARY KEY` already auto-numbers.
- Composite PRIMARY KEY columns can hold NULL in old-style tables unless you add NOT NULL (a SQLite quirk).
- You cannot `ALTER TABLE ... ADD CONSTRAINT`. You rebuild the table instead.
- The PRAGMA to turn foreign keys on/off does nothing inside a transaction.
### When it matters
- Any time more than one program, script, or person writes to the same database.
- When you want bad data to fail loudly at write time, not quietly at report time.
### One-line recap example
```sql
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL REFERENCES customers(id) ON DELETE CASCADE, total REAL CHECK (total >= 0));
```

## 01. PRIMARY KEY and the rowid alias
- Every normal SQLite table has a hidden 64-bit integer `rowid`.
- A column declared exactly `INTEGER PRIMARY KEY` becomes another name for that rowid.
- If you leave it out on INSERT, SQLite picks max(existing) + 1.
- Lookups by it are the fastest possible (no extra index).

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL
);
INSERT INTO departments (dept_name) VALUES ('Engineering'), ('Sales');
SELECT dept_id, dept_name, rowid AS same_as_dept_id FROM departments ORDER BY dept_id;

dept_id | dept_name   | same_as_dept_id
--------+-------------+----------------
1       | Engineering | 1              
2       | Sales       | 2              
(2 rows)



## 02. INTEGER PRIMARY KEY vs other key types
- Only the exact spelling `INTEGER PRIMARY KEY` gets the rowid-alias magic.
- `INT PRIMARY KEY` is NOT an alias: it creates a separate index and does not auto-number.
- A `TEXT PRIMARY KEY` is fine (e.g. a SKU), but remember: it may be NULL in old tables, so add NOT NULL.

In [3]:
%%sql
CREATE TABLE t_int (id INT PRIMARY KEY, v TEXT);
CREATE TABLE t_integer (id INTEGER PRIMARY KEY, v TEXT);
INSERT INTO t_int (v) VALUES ('a');
INSERT INTO t_integer (v) VALUES ('a');
SELECT 'INT' AS declared, id FROM t_int
UNION ALL
SELECT 'INTEGER', id FROM t_integer
ORDER BY declared;

declared | id  
---------+-----
INT      | NULL
INTEGER  | 1   
(2 rows)



## 03. AUTOINCREMENT vs plain rowid
- Plain `INTEGER PRIMARY KEY` picks max(rowid)+1. If you delete the highest row, that number can be reused.
- `AUTOINCREMENT` keeps a counter (in `sqlite_sequence`) so numbers are never reused.
- The price: a little extra bookkeeping on every insert. Use it only when reuse would be a real problem (e.g. ids shown to customers).

In [4]:
%%sql
CREATE TABLE plain_ids (id INTEGER PRIMARY KEY, note TEXT);
CREATE TABLE auto_ids  (id INTEGER PRIMARY KEY AUTOINCREMENT, note TEXT);
INSERT INTO plain_ids (note) VALUES ('one'), ('two'), ('three');
INSERT INTO auto_ids  (note) VALUES ('one'), ('two'), ('three');
DELETE FROM plain_ids WHERE id = 3;
DELETE FROM auto_ids  WHERE id = 3;
INSERT INTO plain_ids (note) VALUES ('new');
INSERT INTO auto_ids  (note) VALUES ('new');
SELECT 'plain' AS kind, id, note FROM plain_ids WHERE note = 'new'
UNION ALL
SELECT 'autoincrement', id, note FROM auto_ids WHERE note = 'new'
ORDER BY kind;
SELECT name, seq FROM sqlite_sequence ORDER BY name;

kind          | id | note
--------------+----+-----
autoincrement | 4  | new 
plain         | 3  | new 
(2 rows)

name     | seq
---------+----
auto_ids | 4  
(1 row)



## 04. Composite PRIMARY KEY
- A key made of several columns: the combination must be unique, each column alone may repeat.
- Classic use: link tables (many-to-many), like student + course.
- It must be written as a table constraint, after the columns.
- Add NOT NULL to each column: SQLite (for historical reasons) lets NULLs into non-INTEGER primary keys.

In [5]:
%%sql
CREATE TABLE enrollments (
    student_id INTEGER NOT NULL,
    course_id  INTEGER NOT NULL,
    grade      TEXT,
    PRIMARY KEY (student_id, course_id)
);
INSERT INTO enrollments VALUES (1, 10, 'A'), (1, 11, 'B'), (2, 10, 'C');
SELECT * FROM enrollments ORDER BY student_id, course_id;

student_id | course_id | grade
-----------+-----------+------
1          | 10        | A    
1          | 11        | B    
2          | 10        | C    
(3 rows)



In [6]:
%%sql expect_error
INSERT INTO enrollments VALUES (1, 10, 'F');

ERROR: UNIQUE constraint failed: enrollments.student_id, enrollments.course_id



## 05. NOT NULL and DEFAULT
- `NOT NULL` rejects any row where the column would be NULL.
- `DEFAULT` supplies a value when the INSERT leaves the column out.
- Gotcha: an explicit NULL is NOT "left out". DEFAULT is not used, and NOT NULL fails.

In [7]:
%%sql
CREATE TABLE tasks (
    task_id INTEGER PRIMARY KEY,
    title   TEXT NOT NULL,
    status  TEXT NOT NULL DEFAULT 'open'
);
INSERT INTO tasks (title) VALUES ('Write notes');
SELECT * FROM tasks ORDER BY task_id;

task_id | title       | status
--------+-------------+-------
1       | Write notes | open  
(1 row)



In [8]:
%%sql expect_error
INSERT INTO tasks (title) VALUES (NULL);
INSERT INTO tasks (title, status) VALUES ('Explicit NULL', NULL);

ERROR: NOT NULL constraint failed: tasks.title

ERROR: NOT NULL constraint failed: tasks.status



## 06. DEFAULT expressions
- A DEFAULT can be a constant, or an expression in parentheses.
- Special keywords: `CURRENT_TIMESTAMP`, `CURRENT_DATE`, `CURRENT_TIME` (evaluated at insert time).
- Here we avoid the clock in the demo by using constants and a computed expression.
- `DEFAULT VALUES` inserts a row where every column uses its default.

In [9]:
%%sql
CREATE TABLE signups (
    signup_id  INTEGER PRIMARY KEY,
    plan       TEXT NOT NULL DEFAULT 'free',
    seats      INTEGER NOT NULL DEFAULT (2 * 5),
    code       TEXT NOT NULL DEFAULT ('ID-' || lower(hex(zeroblob(2)))),
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);
INSERT INTO signups DEFAULT VALUES;
INSERT INTO signups (plan, seats) VALUES ('pro', 50);
SELECT signup_id, plan, seats, code, created_at IS NOT NULL AS has_timestamp FROM signups ORDER BY signup_id;

signup_id | plan | seats | code    | has_timestamp
----------+------+-------+---------+--------------
1         | free | 10    | ID-0000 | 1            
2         | pro  | 50    | ID-0000 | 1            
(2 rows)



## 07. UNIQUE and NULL
- UNIQUE: no two rows may share the same value.
- Unlike PRIMARY KEY, a UNIQUE column may hold NULL, and NULLs never equal each other, so many NULLs are allowed.
- UNIQUE on several columns means the combination is unique.
- Case matters: 'A@x.com' and 'a@x.com' are different unless you use `COLLATE NOCASE`.

In [10]:
%%sql
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY,
    email      TEXT UNIQUE,
    handle     TEXT COLLATE NOCASE UNIQUE
);
INSERT INTO accounts (email, handle) VALUES ('a@x.com', 'Anna');
INSERT INTO accounts (email, handle) VALUES (NULL, 'Ben');
INSERT INTO accounts (email, handle) VALUES (NULL, 'Cy');
SELECT * FROM accounts ORDER BY account_id;

account_id | email   | handle
-----------+---------+-------
1          | a@x.com | Anna  
2          | NULL    | Ben   
3          | NULL    | Cy    
(3 rows)



In [11]:
%%sql expect_error
INSERT INTO accounts (email, handle) VALUES ('a@x.com', 'Dee');
INSERT INTO accounts (email, handle) VALUES ('z@x.com', 'ANNA');

ERROR: UNIQUE constraint failed: accounts.email

ERROR: UNIQUE constraint failed: accounts.handle



## 08. CHECK
- A CHECK is a rule every row must satisfy, written as a boolean expression.
- Great for ranges, allowed values, and rules across columns of one row.
- Gotcha: the row passes if the expression is TRUE **or NULL**. So `CHECK (price >= 0)` still allows NULL price. Add NOT NULL.
- A CHECK cannot look at other rows or other tables (use a trigger or foreign key for that; see 02-Views-And-Triggers).

In [12]:
%%sql
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name       TEXT NOT NULL,
    price      REAL CHECK (price >= 0),
    status     TEXT NOT NULL DEFAULT 'active' CHECK (status IN ('active', 'retired')),
    sale_price REAL,
    CHECK (sale_price IS NULL OR sale_price <= price)
);
INSERT INTO products (name, price) VALUES ('Pen', 1.5);
INSERT INTO products (name, price) VALUES ('Mystery', NULL);   -- NULL passes CHECK!
SELECT * FROM products ORDER BY product_id;

product_id | name    | price | status | sale_price
-----------+---------+-------+--------+-----------
1          | Pen     | 1.5   | active | NULL      
2          | Mystery | NULL  | active | NULL      
(2 rows)



In [13]:
%%sql expect_error
INSERT INTO products (name, price) VALUES ('Bad', -5);
INSERT INTO products (name, price, status) VALUES ('Bad', 5, 'gone');
INSERT INTO products (name, price, sale_price) VALUES ('Bad', 5, 9);

ERROR: CHECK constraint failed: price >= 0

ERROR: CHECK constraint failed: status IN ('active', 'retired')

ERROR: CHECK constraint failed: sale_price IS NULL OR sale_price <= price



## 09. Column vs table constraints, and naming them
- Column constraint: written next to one column. Table constraint: written after all columns; needed when a rule involves several columns.
- Give a constraint a name with `CONSTRAINT name ...`. The name shows up in the error message, which makes debugging much easier.
- Bad: `CHECK constraint failed: products` (which one?). Good: `CHECK constraint failed: ck_price_positive`.

In [14]:
%%sql
CREATE TABLE rooms (
    room_id  INTEGER PRIMARY KEY,
    building TEXT NOT NULL,
    number   INTEGER NOT NULL,
    capacity INTEGER NOT NULL CONSTRAINT ck_capacity_positive CHECK (capacity > 0),
    CONSTRAINT uq_building_number UNIQUE (building, number)
);
INSERT INTO rooms (building, number, capacity) VALUES ('North', 101, 30);

In [15]:
%%sql expect_error
INSERT INTO rooms (building, number, capacity) VALUES ('North', 102, 0);
INSERT INTO rooms (building, number, capacity) VALUES ('North', 101, 20);

ERROR: CHECK constraint failed: ck_capacity_positive

ERROR: UNIQUE constraint failed: rooms.building, rooms.number



## 10. FOREIGN KEY basics and the PRAGMA
- A foreign key says: this column must match an existing key in another (parent) table.
- SQLite parses foreign keys but does NOT enforce them until you run `PRAGMA foreign_keys = ON` on that connection.
- This notebook's connection already turned it on. We will switch it off once to show the silent failure.
- Parent columns must be a PRIMARY KEY or UNIQUE, otherwise you get a "foreign key mismatch" error on use.

In [16]:
%%sql
PRAGMA foreign_keys;
CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    dept_id    INTEGER REFERENCES departments (dept_id)
);
INSERT INTO employees (first_name, dept_id) VALUES ('Alice', 1);
SELECT * FROM employees ORDER BY emp_id;

foreign_keys
------------
1           
(1 row)

emp_id | first_name | dept_id
-------+------------+--------
1      | Alice      | 1      
(1 row)



In [17]:
%%sql expect_error
INSERT INTO employees (first_name, dept_id) VALUES ('Ghost', 999);

ERROR: FOREIGN KEY constraint failed



In [18]:
%%sql
PRAGMA foreign_keys = OFF;
INSERT INTO employees (first_name, dept_id) VALUES ('Ghost', 999);   -- silently accepted
PRAGMA foreign_keys = ON;
SELECT * FROM employees ORDER BY emp_id;

emp_id | first_name | dept_id
-------+------------+--------
1      | Alice      | 1      
2      | Ghost      | 999    
(2 rows)



## 11. Finding violations with foreign_key_check
- `PRAGMA foreign_key_check` lists every row that breaks a foreign key, even with enforcement off.
- Columns: child table, child rowid, parent table, which FK.
- Run it after bulk loads, imports, or table rebuilds.
- Fix the data, then it returns no rows.

In [19]:
%%sql
PRAGMA foreign_key_check;
DELETE FROM employees WHERE dept_id = 999;
PRAGMA foreign_key_check;

table     | rowid | parent      | fkid
----------+-------+-------------+-----
employees | 2     | departments | 0   
(1 row)

table | rowid | parent | fkid
------+-------+--------+-----
(0 rows)



## 12. ON DELETE / ON UPDATE actions
- The action says what happens to child rows when the parent row is deleted (or its key changes).
- `CASCADE`: delete (or update) the children too.
- `SET NULL`: keep the children, blank their FK. The column must allow NULL.
- `SET DEFAULT`: set the FK to its default value.
- `RESTRICT`: refuse the parent change if children exist, immediately.
- `NO ACTION` (the default): also refuses, but the check happens at the end of the statement (or transaction if deferred).

In [20]:
%%sql
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers (customer_id) ON DELETE CASCADE ON UPDATE CASCADE,
    total       REAL NOT NULL
);
CREATE TABLE tickets (
    ticket_id   INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers (customer_id) ON DELETE SET NULL ON UPDATE CASCADE,
    subject     TEXT NOT NULL
);
CREATE TABLE invoices (
    invoice_id  INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers (customer_id) ON DELETE RESTRICT,
    amount      REAL NOT NULL
);
INSERT INTO customers VALUES (1, 'Asha'), (2, 'Ben'), (3, 'Cleo');
INSERT INTO orders   VALUES (1, 1, 20), (2, 1, 35), (3, 2, 10);
INSERT INTO tickets  VALUES (1, 1, 'Late delivery'), (2, 2, 'Refund');
INSERT INTO invoices VALUES (1, 3, 99);
SELECT 'ok' AS status;

status
------
ok    
(1 row)



## 13. CASCADE and SET NULL in action
- Deleting Asha removes her orders (CASCADE) but keeps her ticket with a NULL customer (SET NULL).
- Changing a parent key with ON UPDATE CASCADE rewrites the children's FK automatically.
- Danger: CASCADE can delete a lot. Use it for true "owned" data (order lines), not for shared data.

In [21]:
%%sql
UPDATE customers SET customer_id = 20 WHERE customer_id = 2;   -- ON UPDATE CASCADE on orders
SELECT order_id, customer_id FROM orders ORDER BY order_id;
DELETE FROM customers WHERE customer_id = 1;
SELECT order_id, customer_id FROM orders ORDER BY order_id;
SELECT ticket_id, customer_id, subject FROM tickets ORDER BY ticket_id;

order_id | customer_id
---------+------------
1        | 1          
2        | 1          
3        | 20         
(3 rows)

order_id | customer_id
---------+------------
3        | 20         
(1 row)

ticket_id | customer_id | subject      
----------+-------------+--------------
1         | NULL        | Late delivery
2         | 20          | Refund       
(2 rows)



## 14. RESTRICT and NO ACTION
- Customer 3 has an invoice with `ON DELETE RESTRICT`, so deleting that customer fails.
- `NO ACTION` behaves the same for ordinary statements. The difference only shows with deferred constraints (next section): RESTRICT never waits, NO ACTION can.
- Good default for important records (invoices, payments): refuse deletes instead of cascading.

In [22]:
%%sql expect_error
DELETE FROM customers WHERE customer_id = 3;

ERROR: FOREIGN KEY constraint failed



In [23]:
%%sql
SELECT customer_id, name FROM customers ORDER BY customer_id;

customer_id | name
------------+-----
3           | Cleo
20          | Ben 
(2 rows)



## 15. DEFERRABLE INITIALLY DEFERRED
- Normally a foreign key is checked after each statement.
- `DEFERRABLE INITIALLY DEFERRED` postpones the check to COMMIT.
- Useful when rows reference each other, or you load child rows before the parent inside one transaction.
- If the rule is still broken at COMMIT, the COMMIT fails and you must ROLLBACK (or fix it and COMMIT again).

In [24]:
%%sql
CREATE TABLE authors (author_id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE books (
    book_id   INTEGER PRIMARY KEY,
    author_id INTEGER NOT NULL REFERENCES authors (author_id) DEFERRABLE INITIALLY DEFERRED,
    title     TEXT NOT NULL
);
BEGIN;
INSERT INTO books VALUES (1, 7, 'Child first');     -- parent 7 does not exist yet: allowed for now
INSERT INTO authors VALUES (7, 'Parent second');
COMMIT;
SELECT b.title, a.name FROM books b JOIN authors a ON a.author_id = b.author_id ORDER BY b.book_id;

title       | name         
------------+--------------
Child first | Parent second
(1 row)



In [25]:
%%sql expect_error
BEGIN;
INSERT INTO books VALUES (2, 99, 'Orphan');
COMMIT;
ROLLBACK;
SELECT COUNT(*) AS books_after_rollback FROM books;

ERROR: FOREIGN KEY constraint failed

books_after_rollback
--------------------
1                   
(1 row)



## 16. Self-referencing foreign keys
- A table can point at itself, e.g. employee -> manager. The top of the tree has NULL.
- Combine with `ON DELETE SET NULL` so deleting a manager does not delete the team.
- Walking the tree is done with a recursive CTE (see 02-Querying/03-Subqueries-And-CTEs).

In [26]:
%%sql
CREATE TABLE staff (
    staff_id   INTEGER PRIMARY KEY,
    name       TEXT NOT NULL,
    manager_id INTEGER REFERENCES staff (staff_id) ON DELETE SET NULL
);
INSERT INTO staff VALUES (1, 'Boss', NULL), (2, 'Lead', 1), (3, 'Dev', 2);
DELETE FROM staff WHERE staff_id = 2;
SELECT staff_id, name, manager_id FROM staff ORDER BY staff_id;

staff_id | name | manager_id
---------+------+-----------
1        | Boss | NULL      
3        | Dev  | NULL      
(2 rows)



## 17. Generated columns (VIRTUAL and STORED)
- A generated column is computed from other columns of the same row. You cannot write to it.
- `VIRTUAL` (default): computed each time you read it, takes no space.
- `STORED`: computed on write and saved, takes space but reads are cheap and it can be indexed.
- Great for derived values you never want to get out of sync (totals, full names, JSON extracts).

In [27]:
%%sql
CREATE TABLE line_items (
    item_id    INTEGER PRIMARY KEY,
    qty        INTEGER NOT NULL CHECK (qty > 0),
    unit_price REAL NOT NULL,
    line_total REAL GENERATED ALWAYS AS (qty * unit_price) VIRTUAL,
    label      TEXT GENERATED ALWAYS AS ('x' || qty) STORED
);
INSERT INTO line_items (qty, unit_price) VALUES (3, 2.5), (10, 1.2);
SELECT item_id, qty, unit_price, line_total, label FROM line_items ORDER BY item_id;

item_id | qty | unit_price | line_total | label
--------+-----+------------+------------+------
1       | 3   | 2.5        | 7.5        | x3   
2       | 10  | 1.2        | 12.0       | x10  
(2 rows)



In [28]:
%%sql expect_error
INSERT INTO line_items (qty, unit_price, line_total) VALUES (1, 1, 99);

ERROR: cannot INSERT into generated column "line_total"



## 18. STRICT tables
- Normal SQLite tables use flexible typing: a column's type is only a hint, so `'abc'` can go into an INTEGER column.
- Add `STRICT` after the closing parenthesis and the type is enforced. Allowed types: INT, INTEGER, REAL, TEXT, BLOB, ANY.
- Strict still converts clean text like `'42'` to 42 (type affinity rules), but rejects `'abc'`.
- Prefer STRICT for new tables. Bad vs good below.

In [29]:
%%sql
CREATE TABLE loose  (id INTEGER PRIMARY KEY, qty INTEGER);
CREATE TABLE strict_ (id INTEGER PRIMARY KEY, qty INTEGER) STRICT;
INSERT INTO loose (qty) VALUES ('abc'), ('42');
INSERT INTO strict_ (qty) VALUES ('42');
SELECT qty, typeof(qty) AS type FROM loose ORDER BY id;
SELECT qty, typeof(qty) AS type FROM strict_ ORDER BY id;

qty | type   
----+--------
abc | text   
42  | integer
(2 rows)

qty | type   
----+--------
42  | integer
(1 row)



In [30]:
%%sql expect_error
INSERT INTO strict_ (qty) VALUES ('abc');

ERROR: cannot store TEXT value in INTEGER column strict_.qty



## 19. WITHOUT ROWID tables
- Normally the table is a b-tree keyed by hidden rowid, plus a separate index for your PRIMARY KEY.
- `WITHOUT ROWID` stores the table directly ordered by your PRIMARY KEY: one b-tree, less space, faster key lookups.
- Good for tables with a non-integer or composite key and smallish rows (lookup/link tables).
- Costs: you must declare a PRIMARY KEY (NOT NULL is enforced automatically), there is no `rowid` column, and large rows make it slower.

In [31]:
%%sql
CREATE TABLE country_codes (
    code TEXT PRIMARY KEY,
    name TEXT NOT NULL
) WITHOUT ROWID;
INSERT INTO country_codes VALUES ('IN', 'India'), ('FR', 'France');
SELECT * FROM country_codes ORDER BY code;

code | name  
-----+-------
FR   | France
IN   | India 
(2 rows)



In [32]:
%%sql expect_error
SELECT rowid FROM country_codes;
INSERT INTO country_codes VALUES (NULL, 'Nowhere');

ERROR: no such column: rowid

ERROR: NOT NULL constraint failed: country_codes.code



## 20. ALTER TABLE: what works, what does not
- Works: `RENAME TO`, `RENAME COLUMN`, `ADD COLUMN`, `DROP COLUMN` (3.35+).
- ADD COLUMN limits: no PRIMARY KEY/UNIQUE, and NOT NULL needs a non-NULL DEFAULT.
- DROP COLUMN fails if the column is a key, indexed, used in a constraint, or a view.
- Not supported at all: ADD CONSTRAINT, ALTER COLUMN (change type or add NOT NULL). Use the rebuild recipe in the next section.

In [33]:
%%sql
CREATE TABLE members (member_id INTEGER PRIMARY KEY, name TEXT NOT NULL, phone TEXT);
INSERT INTO members (name, phone) VALUES ('Ravi', '555-0100');
ALTER TABLE members ADD COLUMN tier TEXT NOT NULL DEFAULT 'basic';
ALTER TABLE members RENAME COLUMN phone TO mobile;
SELECT * FROM members ORDER BY member_id;
ALTER TABLE members DROP COLUMN mobile;
SELECT * FROM members ORDER BY member_id;

member_id | name | mobile   | tier 
----------+------+----------+------
1         | Ravi | 555-0100 | basic
(1 row)

member_id | name | tier 
----------+------+------
1         | Ravi | basic
(1 row)



In [34]:
%%sql expect_error
ALTER TABLE members ADD COLUMN age INTEGER NOT NULL;
ALTER TABLE members ADD CONSTRAINT ck CHECK (tier <> '');
ALTER TABLE members ALTER COLUMN name SET NOT NULL;

ERROR: Cannot add a NOT NULL column with default value NULL

ERROR: near "CONSTRAINT": syntax error

ERROR: near "ALTER": syntax error



## 21. The 12-step rebuild (changing a table safely)
- To add a constraint or change a column, build a new table, copy the data, drop the old one, rename the new one.
- Official recipe (shortened): turn foreign keys off, BEGIN, create `new_x`, copy rows with INSERT ... SELECT, DROP old, RENAME new_x to x, recreate indexes/triggers/views, run `PRAGMA foreign_key_check`, COMMIT, turn foreign keys back on.
- Foreign keys must be turned off first, because dropping the old table would otherwise cascade-delete children.
- Do it in a transaction so a failure leaves the original untouched.
- Here: add a CHECK to `members.tier`.

In [35]:
%%sql
PRAGMA foreign_keys = OFF;
BEGIN;
CREATE TABLE members_new (
    member_id INTEGER PRIMARY KEY,
    name      TEXT NOT NULL,
    tier      TEXT NOT NULL DEFAULT 'basic' CHECK (tier IN ('basic', 'gold'))
);
INSERT INTO members_new (member_id, name, tier) SELECT member_id, name, tier FROM members;
DROP TABLE members;
ALTER TABLE members_new RENAME TO members;
PRAGMA foreign_key_check;
COMMIT;
PRAGMA foreign_keys = ON;
SELECT * FROM members ORDER BY member_id;

table | rowid | parent | fkid
------+-------+--------+-----
(0 rows)

member_id | name | tier 
----------+------+------
1         | Ravi | basic
(1 row)



In [36]:
%%sql expect_error
INSERT INTO members (name, tier) VALUES ('Zed', 'platinum');

ERROR: CHECK constraint failed: tier IN ('basic', 'gold')



## 22. Which constraint, which job?
- Row must exist elsewhere -> FOREIGN KEY. Must not repeat -> UNIQUE / PRIMARY KEY.
- Value must be present -> NOT NULL. Value must be in a range or list -> CHECK.
- Value should default sensibly -> DEFAULT. Value derived from others -> generated column.
- Rule needs other rows or tables, or a side effect -> trigger (see 02-Views-And-Triggers).
- Constraints on columns that you filter or join on may also want an index (see 07-Performance/01-Indexes-And-Query-Plans).

In [37]:
%%sql
SELECT m.name AS table_name, p.name AS column_name, p.type, p."notnull", p.pk
FROM sqlite_master m, pragma_table_info(m.name) p
WHERE m.type = 'table' AND m.name IN ('enrollments', 'tasks')
ORDER BY m.name, p.cid;

table_name  | column_name | type    | notnull | pk
------------+-------------+---------+---------+---
enrollments | student_id  | INTEGER | 1       | 1 
enrollments | course_id   | INTEGER | 1       | 2 
enrollments | grade       | TEXT    | 0       | 0 
tasks       | task_id     | INTEGER | 0       | 1 
tasks       | title       | TEXT    | 1       | 0 
tasks       | status      | TEXT    | 1       | 0 
(6 rows)

